In [1]:
import requests
import xmltodict
import xml.etree.ElementTree as ET
from requests.packages.urllib3.exceptions import InsecureRequestWarning
import ssl
import pandas as pd

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
ssl_context = ssl.create_default_context(purpose=ssl.Purpose.CLIENT_AUTH)
ssl_context.verify_mode = ssl.CERT_NONE

user='hscroot'
passwd='Welc0me123'
#HMCname='dcmipphhmc003.edc.nam.gm.com'
HMCnames = [
"dcmidphhmc002.edc.nam.gm.com"
#,"dcmidvmhmc001.edc.nam.gm.com"
#,"dcwitphhmc001.edc.nam.gm.com"
#,"dcwitvmhmc001.edc.nam.gm.com"
#,"dcmirphhmc001.rls.mpg.nam.gm.com"
#,"dcwipphhmc004.edc.nam.gm.com"
#,"dcwipphhmc005.edc.nam.gm.com"
#,"dcmipphhmc002.edc.nam.gm.com"
#,"dcmipphhmc001.edc.nam.gm.com"
#,"dcwipphhmc006.edc.nam.gm.com"
#,"dcwipvmhmc009.edc.nam.gm.com"
#,"dcmipphhmc003.edc.nam.gm.com"
#,"dcmipvmhmc005.edc.nam.gm.com"
#,"dcmipphhmc005.edc.nam.gm.com"
#,"dcmipphhmc006.edc.nam.gm.com"
#,"dcwipphhmc008.edc.nam.gm.com"
#,"dcwipphhmc009.edc.nam.gm.com"
]



def loginHmc(user,passwd,HMCname):
    logonheaders = {'Content-Type': 'application/vnd.ibm.powervm.web+xml; type=LogonRequest'}
    logonUrl =  'https://'+HMCname+':12443/rest/api/web/Logon'
    logonPayload = '<LogonRequest schemaVersion="V1_0" xmlns="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/" xmlns:mc="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/"><UserID>' + user + '</UserID><Password>' + passwd + '</Password></LogonRequest>'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.put(logonUrl,data=logonPayload,headers=logonheaders,proxies=proxies,verify=False)

    if ret.status_code != 200:
        print("Error: Logon failed error code=%d url=%s" %(ret.status_code,logonUrl))
        if self.debug:
                print("DEBUG: Returned:%s" %(ret.text))
        # Do not return if we failed to logon
        sys.exit(ret.status_code)
    xmlResponse = ET.fromstring(ret.text)
    token = xmlResponse[1].text
    return token

def logoffHmc(hmc,token):
    logoffheaders = {'X-API-Session' : token }
    logoffUrl =  'https://'+hmc+':12443/rest/api/web/Logon'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.delete(logoffUrl,headers=logoffheaders,proxies=proxies,verify=False)
    rcode = ret.status_code
    if rcode == 200 or rcode == 202 or rcode == 204:
        print("DEBUG:Successfully disconnected from the HMC ("+hmc+") (code=%d)" %(rcode))
    else:
        print("Error: Logoff failed error code=%d url=%s data=%s" %(rcode, logoffUrl, ret.text))
        sys.exit(rcode)

In [2]:
ManagedSystemName=[]
ManagedSystemFW=[]
ManagedSystemID=[]
ManagedSystemType=[]
HMCInfo=[]
for hmc in HMCnames:
    token=loginHmc(user,passwd,hmc)
    prefHeaders = {'X-API-Session' : token}
    prefUrl = 'https://'+hmc+':12443/rest/api/uom/ManagedSystem'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.get(prefUrl, headers=prefHeaders,proxies=proxies,verify=False).content#.decode('utf-8')
    out=xmltodict.parse(ret)
    for MgmgtSystem in range(0,len(out.get('feed').get('entry'))):
        ManagedSystemName.append(out.get('feed').get('entry')[MgmgtSystem].get('content').get('ManagedSystem:ManagedSystem').get('SystemName').get('#text'))
        ManagedSystemID.append(out.get('feed').get('entry')[MgmgtSystem].get('content').get('ManagedSystem:ManagedSystem').get('Metadata').get('Atom').get('AtomID'))
        ManagedSystemFW.append(out.get('feed').get('entry')[MgmgtSystem].get('content').get('ManagedSystem:ManagedSystem').get('ActivatedServicePackNameAndLevel').get('#text'))
        ManagedSystemType.append(out.get('feed').get('entry')[1].get('content').get('ManagedSystem:ManagedSystem').get('ServiceProcessorVersion').get('#text'))
        HMCInfo.append(hmc)
    logoffHmc(hmc,token)
df = pd.DataFrame(
{
"ManagedSystemName":ManagedSystemName
,"ManagedSystemFW":ManagedSystemFW
,"ManagedSystemID":ManagedSystemID
,"ManagedSystemType":ManagedSystemType
,"HMCInf0":HMCInfo}
)
df["DateVal"]= pd.Timestamp.today().strftime('%Y-%m-%d %H:%M:%S')

df


DEBUG:Successfully disconnected from the HMC (dcmidphhmc002.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcmidvmhmc001.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwitphhmc001.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwitvmhmc001.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcmirphhmc001.rls.mpg.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwipphhmc004.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwipphhmc005.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcmipphhmc002.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcmipphhmc001.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwipphhmc006.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from the HMC (dcwipvmhmc009.edc.nam.gm.com) (code=204)
DEBUG:Successfully disconnected from th

KeyError: 0

In [44]:
#user='hscroot'
#passwd='Welc0me123'
#hmc='dcmipphhmc003.edc.nam.gm.com'
#token=loginHmc(user,passwd,hmc)
#prefHeaders = {'X-API-Session' : token}
#prefUrl = 'https://'+hmc+':12443/rest/api/uom/ManagedSystem'
#proxies = {
#            "http": "",
#            "https": "",
#            }
#ret = requests.get(prefUrl, headers=prefHeaders,proxies=proxies,verify=False).content#.decode('utf-8')
#out=xmltodict.parse(ret)